In [158]:
import networkx as nx
from bp.world import random_grid_world, Scenario, Sequence, Arc

In [159]:
world = random_grid_world(
    rows=4,
    cols=4,
    demands={
        ((0, 0), (3, 3)): 10,
        ((1, 0), (1, 3)): 5,
    },
    seed=0,
)
G = world.network.graph

print(f"{world.total_population=}")
print(f"{world.total_demand((0, 0), (3, 3))=}")
print(f"{world.total_demand((1, 0), (1, 3))=}")

world.total_population=15
world.total_demand((0, 0), (3, 3))=10
world.total_demand((1, 0), (1, 3))=5


In [160]:
from itertools import pairwise
def edge_path(path):
    """Given a path consisting of vertices, return the corresponding edge-path"""
    return list(pairwise(path)) # list for state safety, can remove if careful though

In [161]:
import numpy as np

def debug_scenario(realized, omega, volumes):
    nominal_travel_time = sum(omega.travel_time[a] for a in world.A)
    realized_travel_time = sum(realized.travel_time[a] for a in world.A)
    average_travel_time = realized_travel_time / world.total_population
    average_increased_time = (realized_travel_time - nominal_travel_time) / world.total_population
    print(f"{nominal_travel_time=:.2f}")
    print(f"{realized_travel_time=:.2f}")
    print(f"{average_travel_time=:.2f}")
    print(f"{average_increased_time=:.2f}")

    flow = np.array([volumes.get(arc, 0) for arc in world.ordered_arcs], dtype=float)
    capacity_used = flow / world.network._arc_array("capacity") * 100
    average_capacity_used = np.average(capacity_used)
    max_capacity_used = np.max(capacity_used)
    print(f"{average_capacity_used=:.2f}%")
    print(f"{max_capacity_used=:.2f}%")

    # should sort then bin-search for `congested` b/c sorting but copy paste lazy
    congested = sum(realized.travel_time[a] > omega.travel_time[a] + 1e-9 for a in world.A)
    print(f"arcs whose travel time rose under congestion: {congested}/{len(world.A)}")

    most_congested = sorted(((a, realized.travel_time[a] - omega.travel_time[a]) for a in world.A ), reverse=True, key=lambda x: x[1])
    for i, ((orig, dest), delta) in enumerate(most_congested[:congested], 1):
        print(f"{i}: {orig}->{dest}: +{delta:.2f}")

In [162]:
routes = {
    plr: edge_path(nx.shortest_path(G, plr.demand.origin, plr.demand.destination, weight="travel_time"))
    for plr in world.individuals
}

volumes = {arc: 0 for arc in world.A}
for route in routes.values():
    for arc in route:
        volumes[arc] += 1

realized_concurrent = world.realize(routes)
omega_concurrent = Scenario.from_world("nominal", world)

print("CONCURRENT:")
debug_scenario(realized_concurrent, omega_concurrent, volumes)

CONCURRENT:
nominal_travel_time=295.79
realized_travel_time=3198.29
average_travel_time=213.22
average_increased_time=193.50
average_capacity_used=67.75%
max_capacity_used=780.45%
arcs whose travel time rose under congestion: 9/48
1: (2, 3)->(3, 3): +924.97
2: (0, 3)->(1, 3): +918.69
3: (0, 0)->(0, 1): +540.43
4: (1, 3)->(2, 3): +383.40
5: (0, 1)->(0, 2): +105.80
6: (0, 2)->(0, 3): +18.67
7: (1, 2)->(1, 3): +8.46
8: (1, 1)->(1, 2): +1.56
9: (1, 0)->(1, 1): +0.51


In [163]:
def dynamic_weight(orig, dest, _):
    return current_travel_times[(orig, dest)]

volumes = {arc: 0 for arc in world.A}
routes = {}
for plr in world.individuals:
    current_travel_times = world.network.actual_travel_times(volumes)
    route = edge_path(nx.shortest_path(G, plr.demand.origin, plr.demand.destination, weight=dynamic_weight))
    routes[plr] = route
    for arc in route:
        volumes[arc] += 1

realized_sequential = world.realize(routes)
omega_sequential = Scenario.from_world("nominal", world)

print("SEQUENTIAL:")
debug_scenario(realized_sequential, omega_sequential, volumes)

SEQUENTIAL:
nominal_travel_time=295.79
realized_travel_time=368.71
average_travel_time=24.58
average_increased_time=4.86
average_capacity_used=51.02%
max_capacity_used=312.18%
arcs whose travel time rose under congestion: 21/48
1: (2, 3)->(3, 3): +23.68
2: (0, 0)->(0, 1): +13.83
3: (1, 2)->(1, 3): +8.46
4: (0, 0)->(1, 0): +5.33
5: (2, 2)->(3, 2): +4.30
6: (3, 2)->(3, 3): +3.97
7: (0, 1)->(0, 2): +2.71
8: (2, 1)->(2, 2): +2.57
9: (1, 0)->(1, 1): +1.95
10: (1, 1)->(1, 2): +1.56
11: (0, 3)->(1, 3): +1.47
12: (2, 0)->(2, 1): +1.14
13: (1, 3)->(2, 3): +0.61
14: (0, 2)->(1, 2): +0.50
15: (1, 0)->(2, 0): +0.35
16: (2, 2)->(2, 3): +0.18
17: (1, 2)->(2, 2): +0.18
18: (2, 1)->(3, 1): +0.08
19: (0, 2)->(0, 3): +0.03
20: (1, 1)->(2, 1): +0.01
21: (3, 1)->(3, 2): +0.01
